*Title:* 0_Assembly
*Goal:* Assemble/clean all reads using UGA Reference Genome

*Date Source:* 2016 data available via NCBI (PRJNA704280), 2017 data available upon acceptance.

# 0.1 FAST QC
*Goal* Examine quality of raw reads for both datasets (i.e., 2016 and 2017)

Notes: Steps are shown for 2016 data only, but was run on both datasets

```bash
#For 2016:
module load fastqc/0.12.0
fastqc *.fastq -o . /home/mary2018/projects/rrg-barrett/mary2018/2016_Samples_RAW_Data/RawDataQCReports -t 6

#Download MultiQC Program:
  virtualenv --no-download UpdatedENV
        source UpdatedENV/bin/activate
        pip install --no-index --upgrade pip
        pip install /home/mary2018/multiqc-1.16.dev0-py3-none-any.whl --no-index 

#Run MultiQC (to get all QC Reports in 1 file)
module load python/3.11.2
source UpdatedENV/bin/activate
multiqc . #--ignore "*_I*" --filename "Raw2016MultiQCFinal" #The script. For 2017, had to ignore the Index files hence "--ignore". Not relevant for 2016 samples.
```


# 0.2 Trimming_Filtering
*Goal* Trim and filter the raw reads, then check quality again
Notes: Steps are shown for 2016 data only, but was run on both datasets

``` bash
module load fastp/0.23.4
for R1 in *1.fastq.gz  
do
name=`echo $R1 | sed 's/.1.fastq.gz\+//'`       
R2=${R1%1.fastq.gz}2.fastq.gz
fastp -i $R1 -I $R2 -o trimmed${R1} -O trimmed${R2} --unpaired1 unpaired${R1} --unpaired2 unpaired${R2} --failed_out ${name}_failed.fastq.gz --detect_adapter_for_pe -q 20 -l 50 -g -3
done
```
FASTQC OF trimmed

``` bash
module load fastqc/0.11.9 #Had to load the older version because the new one can't read zipped files!!
fastqc trimmed*.fastq.gz  -o . /home/mary2018/projects/rrg-barrett/mary2018/2016_Samples_RAW_Data/TrimmedDataQCReports -t 6

#MultiQc of trimmed reads (2016)
module load python/3.11.2
source UpdatedENV/bin/activate
multiqc . --filename "Trimmed2017MultiQCFinal" 
```

   # 0.3_Align the genome
   *Notes:* Aligning to UGA, cleaning ambiguous reads

``` bash
# INDEX THE GENOME FOR BWA
#Using UGA v.5 (https://stickleback.genetics.uga.edu/about/)

#Copy from desktop
scp  stickleback_v5.0.1_assembly.fa.gz mary2018@beluga.alliancecan.ca:/home/mary2018/projects/rrg-barrett/mary2018/Reference
gunzip stickleback_v5.0.1_assembly.fa.gz
module load bwa/0.7.17
bwa index stickleback_v5.0.1_assembly.fa

#### STEP4: BWA MEM
module load bwa/0.7.17
for R1 in *1.fastq.gz   
do
name=`echo $R1 | sed 's/.1.fastq.gz\+//'`       
R2=${R1%1.fastq.gz}2.fastq.gz
library=`echo $R1 | sed 's/.*\.\([^.]*\)_.*/\1/'`

bwa mem -M -t 40 -R "@RG\tID:${name}\tPL:ILLUMINA\tLB:${library}\tSM:${library}" /home/mary2018/projects/rrg-barrett/mary2018/Reference/stickleback_v5.0.1_assembly.fa $R1 $R2 > ${name}.sam 2> ${name}.BWAmem.log
done
``` 
Cleaning,sorting, merging:

``` bash
#cleaning/sorting
module load samtools/1.17
for i in *.sam  
do  
echo $i  
name=${i%.sam}
echo $name  
samtools view -@ 10 -q 20 -b -S $i > ${name}_2016cleaned.bam
samtools sort ${name}_2016cleaned.bam -o ${name}_sorted_2016.bam
done

#Checking RG (quality measure)
samtools view -H trimmedHI.4271.006.Index_6.YoungerFA16_.sam | grep '^@RG'

#Merging
#Merging was only needed for the 2016 samples, as the same samples were run over multiple lanes
module load samtools/1.17
samtools merge YNG_SP-2016-merged.bam *sorted_2016.bam
```

# 0.4_Quality
*Goal:* Remove PCR Duplicates, Alignment Stats, bamlist, mpileup generation

PCR Duplicates:
``` bash
#Submitted each population in a separate script to allow for faster processing. 
#Looped script listed below:
module load java/13.0.2
module load picard/2.26.3

for i in *.bam
do  
echo $i  
name=${i%.bam}
echo $name  
java -Xmx150G -XX:-UseGCOverheadLimit -jar $EBROOTPICARD/picard.jar MarkDuplicates -INPUT ${name}.bam -OUTPUT ${name}_dedup.bam -METRICS_FILE ${name}_dup_metrics.txt -REMOVE_DUPLICATES true
done
```

Quality/Alignment Stats per bam file:

``` bash
module load samtools/1.17
(for i in *.bam
do 
samtools flagstat $i
done) > out2016.txt
``` 
Create a bamlist:
#Copied dedup.bam from 2016+2017 to the same directory
```  bash
for f in `ls -S *.bam`
do echo $f >> bam_list.txt
done
``` 
SAMTOOLS MPILEUP
``` bash
samtools mpileup -B -f /home/mary2018/projects/rrg-barrett/mary2018/Reference/stickleback_v5.0.1_assembly.fa -b bam_list.txt > finalSCmulti.mpileup
```

#0.5_Cleaning_Sync
*Goal:* indentify indents, filter for completness, create sync, mask indels, filter for autosomes only. 


IDENTIFY INDELS
``` bash
perl /home/mary2018/.local/bin/popoolation2_1201/indel_filtering/identify-indel-regions.pl --input finalSCmulti.mpileup --output indel-regions.gtf --indel-window 5 --min-count 1
``` 
FILTER FOR COMPLETNESS

Goal: 
#My version which explicitely ignores the second column (position) when searching for matches from 1-5. Without it, I think that positions 1-5 will be removed (rather than just depths of 1-5). 
``` bash
awk 'BEGIN {OFS="\t"} {
    flag = 0
    for (i = 1; i <= NF; i++) {
        if (i != 2 && int($i) == $i && $i >= 0 && $i <= 5) {
            flag = 1
            break
        }
    }
    if (flag == 0) {
        print
    }
}' finalSCmulti.mpileup > SCMulti-MKattempt2Filtered.mpileup
``` 
CREATE A SYNC FILE
``` bash
module load java
java -ea -Xmx2g -jar /home/mary2018/.local/bin/popoolation2_1201/mpileup2sync.jar --input SCMulti-MKattempt2Filtered.mpileup --output 2hopefully-actual-final-javaSCMulti-MKattempt2Filtered.sync --fastq>
```
Filtering the indels
``` bash
perl /home/mary2018/.local/bin/popoolation2_1201/indel_filtering/filter-sync-by-gtf.pl --gtf indel-regionsmkattempt2filtered.gtf --input 2hopefully-actual-final-javaSCMulti-MKattempt2Filtered.sync --output maskedindels-2hopefully-actual-final-javaSCMulti-MKattempt2Filtered.sync
```

#Removing scaffolds
Goal: remove mitochondrial, scaffolds, and sex chromosome
``` bash

grep -v -E '^(chrY|chrM|chrUn|chrXIX)\s' maskedindels-2hopefully-actual-final-javaSCMulti-MKattempt2Filtered.sync > masked-autosome-chrom.sync
``` 